In [1]:
import numpy as np
from cp import run_split_conformal

experiment_name = "resnet18_224_scratch"
path = f"./outputs/{experiment_name}_outputs.npz"

data = np.load(path)

val_probs = data["val_probs"]
val_labels = data["val_labels"]
test_probs = data["test_probs"]
test_labels = data["test_labels"]

for alpha in [0.1, 0.05]:
    result = run_split_conformal(
        cal_probs=val_probs,
        cal_labels=val_labels,
        test_probs=test_probs,
        test_labels=test_labels,
        alpha=alpha,
    )

    # 🔴 BURASI: qhat ve scores hesaplandıktan sonra
    qhat = result["qhat"]

    val_true_probs = val_probs[np.arange(len(val_labels)), val_labels]
    test_true_probs = test_probs[np.arange(len(test_labels)), test_labels]

    val_scores = 1.0 - val_true_probs
    test_scores = 1.0 - test_true_probs

    print(f"\n=== {experiment_name} | alpha={alpha} ===")
    print(f"qhat: {qhat:.6f}")
    print(result["metrics"])

    print("Calibration coverage wrt qhat:", (val_scores <= qhat).mean())
    print("Test coverage wrt qhat:", (test_scores <= qhat).mean())

    print("Val class counts :", np.bincount(val_labels))
    print("Test class counts:", np.bincount(test_labels))

    print("min prob:", test_probs.min())
    print("max prob:", test_probs.max())


=== resnet18_224_scratch | alpha=0.1 ===
qhat: 0.283020
{'coverage': 0.707, 'avg_set_size': 0.868, 'singleton_rate': 0.868, 'empty_set_rate': 0.132, 'class_wise_coverage': {0: 0.98, 1: 0.716, 2: 0.28, 3: 0.852}}
Calibration coverage wrt qhat: 0.9002031019202363
Test coverage wrt qhat: 0.707
Val class counts : [3721 1135  862 5114]
Test class counts: [250 250 250 250]
min prob: 8.465135e-11
max prob: 1.0

=== resnet18_224_scratch | alpha=0.05 ===
qhat: 0.688961
{'coverage': 0.804, 'avg_set_size': 1.077, 'singleton_rate': 0.923, 'empty_set_rate': 0.0, 'class_wise_coverage': {0: 0.992, 1: 0.84, 2: 0.44, 3: 0.944}}
Calibration coverage wrt qhat: 0.9502400295420975
Test coverage wrt qhat: 0.804
Val class counts : [3721 1135  862 5114]
Test class counts: [250 250 250 250]
min prob: 8.465135e-11
max prob: 1.0


In [2]:
import numpy as np
from cp import run_split_conformal, run_class_conditional_conformal

experiment_name = "resnet18_224_scratch"
path = f"./outputs/{experiment_name}_outputs.npz"

data = np.load(path)

val_probs = data["val_probs"]
val_labels = data["val_labels"]
test_probs = data["test_probs"]
test_labels = data["test_labels"]

for alpha in [0.1, 0.05]:
    print(f"\n==============================")
    print(f"{experiment_name} | alpha={alpha}")
    print(f"==============================")

    # Standard split conformal
    result_std = run_split_conformal(
        cal_probs=val_probs,
        cal_labels=val_labels,
        test_probs=test_probs,
        test_labels=test_labels,
        alpha=alpha,
    )

    print("\n--- Standard split conformal ---")
    print("qhat:", result_std["qhat"])
    print(result_std["metrics"])

    # Class-conditional conformal
    result_cc = run_class_conditional_conformal(
        cal_probs=val_probs,
        cal_labels=val_labels,
        test_probs=test_probs,
        test_labels=test_labels,
        alpha=alpha,
    )

    print("\n--- Class-conditional conformal ---")
    print("qhats:", result_cc["qhats"])
    print(result_cc["metrics"])


resnet18_224_scratch | alpha=0.1

--- Standard split conformal ---
qhat: 0.2830202579498291
{'coverage': 0.707, 'avg_set_size': 0.868, 'singleton_rate': 0.868, 'empty_set_rate': 0.132, 'class_wise_coverage': {0: 0.98, 1: 0.716, 2: 0.28, 3: 0.852}}

--- Class-conditional conformal ---
qhats: {0: 0.06875920295715332, 1: 0.6432318985462189, 2: 0.9659320265054703, 3: 0.18344438076019287}
{'coverage': 0.838, 'avg_set_size': 1.035, 'singleton_rate': 0.899, 'empty_set_rate': 0.033, 'class_wise_coverage': {0: 0.944, 1: 0.828, 2: 0.788, 3: 0.792}}

resnet18_224_scratch | alpha=0.05

--- Standard split conformal ---
qhat: 0.6889610886573792
{'coverage': 0.804, 'avg_set_size': 1.077, 'singleton_rate': 0.923, 'empty_set_rate': 0.0, 'class_wise_coverage': {0: 0.992, 1: 0.84, 2: 0.44, 3: 0.944}}

--- Class-conditional conformal ---
qhats: {0: 0.259279727935791, 1: 0.9447218850255013, 2: 0.9882642105221748, 3: 0.3871247172355652}
{'coverage': 0.919, 'avg_set_size': 1.291, 'singleton_rate': 0.737, 'e